# Notebook 13 — Quarantine por RElabel (E26) e submissão sub10

Substitui o *drop-worst* do Notebook 11/12 por **relabel**: as piores 15% das linhas
(rank por confiança própria no OOF cross-fit do `nb11_oof311.npz`) passam a ter o rótulo
substituído pela previsão OOF do XGB. Nenhuma linha é descartada; a validação usa sempre
os rótulos originais. Fonte do relabel limpa (OOF sem seleção dependente do rótulo).

Protocolo igual ao NB12: `StratifiedKFold(5, shuffle, seed=42)` em `train_nb02.csv`,
Macro-F1, receita Frank&Hall 5 sementes, correção `max(pred, FAIXA_MINIMA)`.
Modo rápido: `NB13_MODO=rapido` (2 folds, 2 sementes, caches `_rap`).

In [1]:
import os, time, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
K = 16
TARGET = 'AGE_RANGE'

MODO = os.environ.get('NB13_MODO', 'completo')
RAPIDO = MODO in ('rapido', 'ecra')
SEEDS = [42, 1] if RAPIDO else [42, 1, 2, 3, 4]
N_SPLITS = 2 if RAPIDO else 5
PCT = int(os.environ.get('NB13_PCT', 15))          # dose do relabel
SUF = '_rap' if RAPIDO else ''


def _data_dir():
    for cand in [os.path.join('..', '..', 'data'), os.path.join('..', 'data'), 'data']:
        if os.path.isdir(cand):
            return cand
    raise FileNotFoundError('pasta data/ nao encontrada')


DATA_DIR = _data_dir()
SUB_DIR = os.path.join(os.path.dirname(DATA_DIR), 'submissions')

train = pd.read_csv(os.path.join(DATA_DIR, 'train_nb02.csv'))
y = train[TARGET].to_numpy().astype(int)
Xv = train.drop(columns=['PID', TARGET]).to_numpy(dtype=np.float64)
fmin = train['FAIXA_MINIMA'].to_numpy().astype(int)
FOLDS = list(StratifiedKFold(N_SPLITS, shuffle=True, random_state=RANDOM_STATE).split(Xv, y))
print('MODO=', MODO, '| X:', Xv.shape, '| dose relabel:', f'{PCT}%', '| seeds:', SEEDS)


def xgb_bin(seed=RANDOM_STATE):
    return XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
        min_child_weight=3, subsample=0.8, colsample_bytree=0.5,
        tree_method='hist', random_state=seed, n_jobs=-1, eval_metric='logloss')

MODO= completo | X: (10962, 311) | dose relabel: 15% | seeds: [42, 1, 2, 3, 4]


## 1. Alvo de relabel (cross-fit, sem seleção por rótulo)

In [2]:
d = np.load(os.path.join(DATA_DIR, 'nb11_oof311.npz'))
oof_x, proba_x = d['oof'].astype(int), d['proba']
rank = np.argsort(proba_x[np.arange(len(y)), y])
rr = rank[:int(len(y) * PCT / 100)]
y_lab = y.copy()
y_lab[rr] = oof_x[rr]
n_mud = int((y_lab != y).sum())
print(f'relabel: {len(rr)} linhas alvo ({PCT}%), {n_mud} rótulos alterados')
print('por classe (true -> novo):')
mud = pd.crosstab(pd.Series(y[rr], name='original'), pd.Series(y_lab[rr], name='novo'))
print(mud.to_string())

relabel: 1644 linhas alvo (15%), 1644 rótulos alterados
por classe (true -> novo):
novo      0   1   2   3   4   5   6   7   8   9   10  11  12  13  14  15
original                                                                
0          0   3   1   1   0   0   0   0   1   0   0   0   0   0   0   0
1         25   0  36   2   1   1   1   0   0   0   0   0   0   0   0   0
2          5  12   0  15  11   7   7   1   2   0   0   0   0   0   0   0
3          2   3  25   0  28  25  16   3   5   0   0   0   0   0   0   0
4          0   2  15  23   0  46  28  19   5   0   0   0   0   0   0   0
5          0   1   4  20  19   0  42  26   8   0   0   0   0   0   0   0
6          0   0   5  11  11  29   0  42  38   0   0   0   0   0   0   0
7          0   0   3   7  11  23  30   0  42   0   0   0   0   0   0   0
8          0   1   1   0   6   7  36  43   0   0   0   0   0   0   0   0
9          0   0   0   0   0   0   0   0   0   0  60   3   0   0   0   1
10         0   0   0   0   0   0   0   1 

## 2. CV Frank&Hall 5 sementes com y_lab (cache `data/nb13_rel{PCT}_5seed{SUF}.npz`)

In [3]:
def fh_cv():
    p = os.path.join(DATA_DIR, f'nb13_rel{PCT}_5seed{SUF}.npz')
    if os.path.exists(p):
        print('cache:', p)
        return np.load(p)['Pall']
    t0 = time.time()
    Pall = np.zeros((len(SEEDS), len(y), K))
    drop = set()                      # relabel: nada é descartado
    for fi, (a, b) in enumerate(FOLDS):
        atr = np.array([i for i in a if i not in drop])
        sw = compute_sample_weight('balanced', y_lab[atr])
        for si, sd in enumerate(SEEDS):
            pg = np.zeros((len(b), K - 1))
            for k in range(K - 1):
                m = xgb_bin(sd)
                m.fit(Xv[atr], (y_lab[atr] > k).astype(int), sample_weight=sw)
                pg[:, k] = m.predict_proba(Xv[b])[:, 1]
            pg = np.minimum.accumulate(pg, axis=1)
            prev = np.ones(len(b))
            for k in range(K):
                cur = pg[:, k] if k < K - 1 else np.zeros(len(b))
                Pall[si, b, k] = np.clip(prev - cur, 0, None)
                prev = cur
        print(f'  fold {fi} ({time.time()-t0:.0f}s)', flush=True)
    np.savez_compressed(p, Pall=Pall)
    return Pall


Pall = fh_cv()
Pmean = Pall.mean(axis=0)


def med_of(p):
    return np.array([int(np.median(np.repeat(np.arange(K), (p[i] * 1000).astype(int))))
                     for i in range(len(p))])


def variantes(Pm, fmin_arr):
    """Regras pré-declaradas: argmax, argmax+corr, mediana@0.40+corr."""
    arg = Pm.argmax(axis=1)
    med = med_of(Pm)
    return {
        'argmax': arg,
        'argmax+corr': np.maximum(arg, fmin_arr),
        'mediana@0.40+corr': np.maximum(np.where(Pm.max(axis=1) < 0.40, med, arg), fmin_arr),
    }


def por_folds(pred):
    return np.array([f1_score(y[va], pred[va], average='macro') for _, va in FOLDS])

cache: ../data/nb13_rel15_5seed.npz


## 3. E26 vs referência E25 (mesmas folds, mesmas regras)

In [4]:
e25 = np.load(os.path.join(DATA_DIR, 'nb12_e25_fh5.npz'))
P25, oof25 = e25['Pmean'], e25['oof'].astype(int)
med25 = med_of(P25)
ref = {
    'argmax': oof25,
    'argmax+corr': np.maximum(oof25, fmin),
    'mediana@0.40+corr': np.maximum(np.where(P25.max(axis=1) < 0.40, med25, oof25), fmin),
}
linhas = []
for nome in ['argmax', 'argmax+corr', 'mediana@0.40+corr']:
    f_e26 = por_folds(variantes(Pmean, fmin)[nome])
    f_e25 = por_folds(ref[nome])
    linhas.append({'regra': nome,
                   'E25_mean': round(f_e25.mean(), 4), 'E25_agg': round(f1_score(y, ref[nome], average='macro'), 4),
                   'E26_mean': round(f_e26.mean(), 4), 'E26_agg': round(f1_score(y, variantes(Pmean, fmin)[nome], average='macro'), 4),
                   'delta_mean': round(f_e26.mean() - f_e25.mean(), 4),
                   'folds_ganhos': int((f_e26 > f_e25).sum())})
res = pd.DataFrame(linhas)
print(res.to_string(index=False))

vencedora = res.sort_values('E26_mean', ascending=False).iloc[0]['regra']
print('regra vencedora no CV:', vencedora)
f1s_v = por_folds(variantes(Pmean, fmin)[vencedora])
print('folds E26:', np.round(f1s_v, 4).tolist())

            regra  E25_mean  E25_agg  E26_mean  E26_agg  delta_mean  folds_ganhos
           argmax    0.5195   0.5197    0.5353   0.5355      0.0158             5
      argmax+corr    0.5207   0.5209    0.5364   0.5365      0.0157             5
mediana@0.40+corr    0.5227   0.5230    0.5353   0.5354      0.0126             5
regra vencedora no CV: argmax+corr


folds E26: [0.5464, 0.546, 0.5324, 0.5284, 0.5288]


### 3.1 Diagnóstico por classe (E26 vencedora vs E25 vencedora)

In [5]:
def per_class(pred):
    r = []
    for c in range(K):
        tp = ((y == c) & (pred == c)).sum(); fp = ((y != c) & (pred == c)).sum()
        fn = ((y == c) & (pred != c)).sum()
        pr, rc = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
        r.append(2 * pr * rc / max(pr + rc, 1e-9))
    return np.array(r)


pc26 = per_class(variantes(Pmean, fmin)[vencedora])
pc25 = per_class(ref[vencedora])
tab = pd.DataFrame({'classe': range(K), 'E25': pc25.round(3), 'E26': pc26.round(3),
                    'delta': (pc26 - pc25).round(3)})
print(tab.to_string(index=False))

RESULTADOS = []
for nome in ['argmax', 'argmax+corr', 'mediana@0.40+corr']:
    f_e26 = por_folds(variantes(Pmean, fmin)[nome])
    f_e25 = por_folds(ref[nome])
    RESULTADOS.append({'exp': 'E26', 'variante': f'rel{PCT}% {nome}',
                       'f1': round(f_e26.mean(), 4), 'std': round(f_e26.std(), 4),
                       'delta_vs_E25': round(f_e26.mean() - f_e25.mean(), 4),
                       'folds_ganhos': int((f_e26 > f_e25).sum())})

 classe   E25   E26  delta
      0 0.872 0.869 -0.003
      1 0.457 0.489  0.032
      2 0.641 0.658  0.016
      3 0.435 0.492  0.057
      4 0.363 0.393  0.029
      5 0.404 0.434  0.030
      6 0.406 0.427  0.022
      7 0.436 0.446  0.010
      8 0.677 0.678  0.000
      9 0.700 0.705  0.005
     10 0.689 0.689 -0.000
     11 0.566 0.569  0.003
     12 0.411 0.425  0.014
     13 0.341 0.360  0.018
     14 0.415 0.435  0.020
     15 0.520 0.518 -0.002


## 4. Previsões de teste (cache por semente `data/nb13_rel{PCT}_te_s{sd}.npz`)

In [6]:
te = pd.read_csv(os.path.join(DATA_DIR, 'test_nb02.csv'))
Xte = te.drop(columns=['PID']).to_numpy(dtype=np.float64)
fmin_te = te['FAIXA_MINIMA'].to_numpy().astype(int)
assert len(Xte) == 3655
assert list(te.drop(columns=['PID']).columns) == list(train.drop(columns=['PID', TARGET]).columns)

sw_lab = compute_sample_weight('balanced', y_lab)
Pte_all = np.zeros((len(SEEDS), len(Xte), K))
t0 = time.time()
for si, sd in enumerate(SEEDS):
    ps = os.path.join(DATA_DIR, f'nb13_rel{PCT}_te_s{sd}{SUF}.npz')
    if os.path.exists(ps):
        Pte_all[si] = np.load(ps)['pf']
        print(f'seed {sd} teste: cache')
        continue
    pg = np.zeros((len(Xte), K - 1))
    for k in range(K - 1):
        m = xgb_bin(sd)
        m.fit(Xv, (y_lab > k).astype(int), sample_weight=sw_lab)
        pg[:, k] = m.predict_proba(Xte)[:, 1]
    pg = np.minimum.accumulate(pg, axis=1)
    prev = np.ones(len(Xte))
    pf = np.zeros((len(Xte), K))
    for k in range(K):
        cur = pg[:, k] if k < K - 1 else np.zeros(len(Xte))
        pf[:, k] = np.clip(prev - cur, 0, None)
        prev = cur
    np.savez_compressed(ps, pf=pf)
    Pte_all[si] = pf
    print(f'seed {sd} teste done ({time.time()-t0:.0f}s)', flush=True)
Pte = Pte_all.mean(axis=0)

seed 42 teste: cache
seed 1 teste: cache
seed 2 teste: cache
seed 3 teste: cache
seed 4 teste: cache


## 5. Submissão sub10 (mesma regra vencedora no CV)

In [7]:
variantes_te = {
    'argmax': Pte.argmax(axis=1),
    'argmax+corr': np.maximum(Pte.argmax(axis=1), fmin_te),
    'mediana@0.40+corr': np.maximum(
        np.where(Pte.max(axis=1) < 0.40, med_of(Pte), Pte.argmax(axis=1)), fmin_te),
}
pred = variantes_te[vencedora].astype(int)
assert len(pred) == 3655 and pred.min() >= 0 and pred.max() <= 15
out = os.path.join(SUB_DIR, 'sub10_fh5_rel15.csv')
pd.DataFrame({'RowId': np.arange(len(pred)), TARGET: pred}).to_csv(out, index=False)
print('gravado:', out)
print('distribuicao previsoes:', pd.Series(pred).value_counts().sort_index().to_dict())

final = pd.DataFrame(RESULTADOS)
final.to_csv(os.path.join(DATA_DIR, f'nb13_resultados{SUF}.csv'), index=False)
print(final.to_string(index=False))

gravado: ../submissions/sub10_fh5_rel15.csv
distribuicao previsoes: {0: 69, 1: 45, 2: 137, 3: 155, 4: 175, 5: 195, 6: 291, 7: 292, 8: 268, 9: 363, 10: 363, 11: 318, 12: 366, 13: 271, 14: 224, 15: 123}
exp                 variante     f1    std  delta_vs_E25  folds_ganhos
E26            rel15% argmax 0.5353 0.0077        0.0158             5
E26       rel15% argmax+corr 0.5364 0.0081        0.0157             5
E26 rel15% mediana@0.40+corr 0.5353 0.0090        0.0126             5
